# sigstore_rekor_types smoke test

Trail of Bits' pydantic v2 models generated from Rekor's OpenAPI schemas (hashedrekord,
dsse, intoto, jar, rekord, rfc3161, rpm, tuf, ...). Pure python.

In [ ]:
import base64, hashlib
import rekor_types

hr = rekor_types.hashedrekord
entry = rekor_types.Hashedrekord(
    spec=hr.HashedrekordV001Schema(
        signature=hr.Signature(public_key=hr.PublicKey(content=base64.b64encode(b"key").decode())),
        data=hr.Data(hash=hr.Hash(algorithm=hr.Algorithm.SHA256,
                                  value=hashlib.sha256(b"artifact").hexdigest())),
    ),
)
as_json = entry.model_dump_json(by_alias=True)
assert '"apiVersion"' in as_json and '"publicKey"' in as_json  # aliased keys on the wire
back = rekor_types.Hashedrekord.model_validate_json(as_json)
assert back == entry
print("round-tripped", len(as_json), "bytes")

In [ ]:
import hashlib
import rekor_types
from pydantic import TypeAdapter, ValidationError

adapter = TypeAdapter(rekor_types.ProposedEntry)
raw = {
    "kind": "hashedrekord",
    "apiVersion": "0.0.1",
    "spec": {"signature": {}, "data": {"hash": {"algorithm": "sha256",
             "value": hashlib.sha256(b"artifact").hexdigest()}}},
}
parsed = adapter.validate_python(raw)
assert isinstance(parsed, rekor_types.Hashedrekord)

try:
    adapter.validate_python({"kind": "not-a-rekor-kind", "apiVersion": "0.0.1", "spec": {}})
    raise AssertionError("unknown kind must be rejected")
except ValidationError as e:
    print("parsed:", type(parsed).__name__, "| unknown kind rejected:", e.error_count(), "error(s)")

In [ ]:
import rekor_types
from pydantic import ValidationError

err = rekor_types.Error(code=404, message="entry not found")
assert err.code == 404 and err.message == "entry not found"

hr = rekor_types.hashedrekord
spec = hr.HashedrekordV001Schema(signature=hr.Signature(), data=hr.Data())  # all-optional inner fields
try:
    rekor_types.Hashedrekord(api_version="not-semver", spec=spec)
    raise AssertionError("non-semver apiVersion must be rejected")
except ValidationError as e:
    print("Error model OK | semver pattern enforced:", e.error_count(), "error(s)")

In [ ]:
import rekor_types

schema = rekor_types.Hashedrekord.model_json_schema(by_alias=True)
assert schema["properties"]["kind"]["const"] == "hashedrekord"
assert schema["required"] == ["spec"]  # kind + apiVersion carry defaults
assert "HashedrekordV001Schema" in schema["$defs"]
print("schema properties:", sorted(schema["properties"]), "| $defs:", len(schema["$defs"]))